# Бейзлайн кейса Astramis

Ноутбук проходит весь путь от кадра до файла с ответами:

1. **Детектор.** YOLO11n-seg находит объекты и определяет форму: сыпучее или цельное. **Материал бейзлайн не определяет** и всегда называет самый частый в обучении.
2. **Объём.** По карте глубины считаем высоту точек над полом внутри маски объекта и складываем её в объём.
3. **Масса и решение.** Масса = объём × плотность из `densities.csv`. Если масса не больше 500 г, робот забирает объект.

Бейзлайн намеренно простой: он не различает материалы, учится только на реальных кадрах и считает объём «в лоб». Каждый шаг можно улучшать отдельно, идеи есть в конце ноутбука.

**Перед запуском:** включите GPU (Среда выполнения → Сменить тип среды → T4) и положите рядом папку `dataset` и файл `score.py`.

In [ ]:
# В Colab раскомментируйте следующую строку:
# !pip -q install ultralytics

In [ ]:
import csv
import json
import random
import shutil
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO

import score as metric                      # файл score.py из архива

DATA = Path("dataset")                      # папка с данными
WORK = Path("work")                         # сюда пишем всё промежуточное
EPOCHS = 10                                 # эпох обучения детектора
IMGSZ = 640                                 # размер картинки на входе детектора
CONF = 0.10                                 # порог уверенности детектора
HEAVY_G = 500                               # тяжелее этого конвейер не берёт

random.seed(0)
CAM = json.loads((DATA / "camera.json").read_text(encoding="utf-8"))


def color_path(frame_dir: Path) -> Path:
    """Путь к цветному кадру (png или jpg)."""
    p = frame_dir / "color.png"
    return p if p.exists() else frame_dir / "color.jpg"

## 1. Читаем разметку и откладываем валидацию

Тест состоит из предметов, которых нет в обучении. Чтобы наша проверка была честной, валидацию собираем так же:
откладываем **целые предметы**, а не случайные кадры. Иначе модель «узнает» знакомую бутылку и балл будет завышен.

In [ ]:
def load(labels_file):
    data = json.loads((DATA / "labels" / labels_file).read_text(encoding="utf-8"))
    anns = defaultdict(list)
    for a in data["annotations"]:
        anns[a["image_id"]].append(a)
    return [{"dir": DATA / Path(im["file_name"]).parent, "id": Path(im["file_name"]).parent.name,
             "w": im["width"], "h": im["height"], "anns": anns[im["id"]]}
            for im in data["images"] if color_path(DATA / Path(im["file_name"]).parent).exists()]


real = load("real_train.json")                               # синтетику (synth_train.json) бейзлайн не использует

objects = sorted({a["attributes"]["object_id"] for f in real for a in f["anns"]})
val_objects = set(objects[::5])                              # каждый пятый предмет уходит в валидацию
is_val = lambda f: any(a["attributes"]["object_id"] in val_objects for a in f["anns"])
val = [f for f in real if is_val(f)]
train = [f for f in real if not is_val(f)]

cls_name = lambda a: a["attributes"]["form"]                 # детектор учим только форме
CLASSES = ["bulk", "solid"]
materials = [a["attributes"]["material"] for f in train for a in f["anns"]]
MATERIAL_GUESS = max(set(materials), key=materials.count)    # самый частый материал в обучении
print(f"Обучение: {len(train)} кадров, валидация: {len(val)}")
print("Классы детектора:", CLASSES, "| материал по умолчанию:", MATERIAL_GUESS)

## 2. Готовим данные для YOLO и обучаем детектор

In [ ]:
yolo_dir = WORK / "yolo"
if yolo_dir.exists():
    shutil.rmtree(yolo_dir)
for split, frames in (("train", train), ("val", val)):
    (yolo_dir / "images" / split).mkdir(parents=True)
    (yolo_dir / "labels" / split).mkdir(parents=True)
    for f in frames:
        src = color_path(f["dir"])
        shutil.copy(src, yolo_dir / "images" / split / f"{f['id']}{src.suffix}")
        lines = []
        for a in f["anns"]:
            if cls_name(a) not in CLASSES:
                continue
            seg = max(a["segmentation"], key=len)            # самый большой контур объекта
            pts = np.array(seg, dtype=float).reshape(-1, 2) / [f["w"], f["h"]]
            lines.append(f"{CLASSES.index(cls_name(a))} " + " ".join(f"{v:.5f}" for v in pts.clip(0, 1).reshape(-1)))
        (yolo_dir / "labels" / split / f"{f['id']}.txt").write_text("\n".join(lines))

(yolo_dir / "data.yaml").write_text(
    f"path: {yolo_dir.resolve()}\ntrain: images/train\nval: images/val\nnames:\n"
    + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASSES)))

In [ ]:
model = YOLO("yolo11n-seg.pt")
model.train(data=str(yolo_dir / "data.yaml"), epochs=EPOCHS, imgsz=IMGSZ,
            project=str(WORK.resolve()), name="seg", exist_ok=True, plots=False, verbose=False)
model = YOLO(str(WORK / "seg" / "weights" / "best.pt"))

## 3. Объём по карте глубины

Каждый пиксель внутри маски переводим в точку 3D и считаем её высоту над плоскостью пола (плоскость записана в `meta.json` кадра).
Пол делим на клетки 4×4 мм, в каждой берём самую высокую точку, и складываем высоты, умноженные на площадь клетки.

In [ ]:
def raw_volume_ml(depth_mm, mask, plane, cell=0.004):
    v, u = np.nonzero(mask & (depth_mm > 0))
    if len(v) < 30:
        return None                                          # глубины почти нет (стекло, блестящий металл)
    z = depth_mm[v, u] / 1000.0
    pts = np.stack([(u - CAM["cx"]) / CAM["fx"] * z, (v - CAM["cy"]) / CAM["fy"] * z, z], axis=1)
    n, d = np.array(plane[:3]), plane[3]
    height = pts @ n + d
    e1 = np.cross(n, [1.0, 0.0, 0.0])
    e1 /= np.linalg.norm(e1)
    e2 = np.cross(n, e1)
    keys = np.floor(pts @ e1 / cell).astype(np.int64) * 100000 + np.floor(pts @ e2 / cell).astype(np.int64)
    order = np.argsort(keys)
    keys, height = keys[order], height[order]
    starts = np.r_[0, np.nonzero(np.diff(keys))[0] + 1]
    tops = np.maximum.reduceat(height, starts)
    return float(np.clip(tops, 0, None).sum() * cell * cell * 1e6)


def polygon_mask(segmentation, h, w):
    m = np.zeros((h, w), np.uint8)
    for seg in segmentation:
        cv2.fillPoly(m, [np.array(seg, np.float32).reshape(-1, 2).round().astype(np.int32)], 1)
    return m > 0


def read_frame(frame_dir):
    depth = cv2.imread(str(frame_dir / "depth.png"), cv2.IMREAD_UNCHANGED)
    plane = json.loads((frame_dir / "meta.json").read_text(encoding="utf-8"))["floor_plane"]
    return depth, plane

## 4. Плотности и запасной объём

Плотности берём из `densities.csv`. Если у объекта нет глубины, подставляем средний по обучению объём.

In [ ]:
TYPICAL_VOLUME = float(np.median([a["attributes"]["volume_ml"] for f in train for a in f["anns"]]))
with open(DATA / "densities.csv", encoding="utf-8") as fh:
    DENSITY = {f"{r['form']}_{r['material']}": float(r["density_g_per_ml"]) for r in csv.DictReader(fh)}
print(f"Запасной объём: {TYPICAL_VOLUME:.0f} мл")

## 5. Предсказание для одного кадра

In [ ]:
def predict(frame_dir):
    """Возвращает список объектов кадра в формате метрики."""
    result = model(str(color_path(frame_dir)), conf=CONF, imgsz=IMGSZ, verbose=False)[0]
    if result.masks is None:
        return []
    depth, plane = read_frame(frame_dir)
    h, w = depth.shape
    out = []
    for box, conf, cls, poly in zip(result.boxes.xyxy.tolist(), result.boxes.conf.tolist(),
                                    result.boxes.cls.tolist(), result.masks.xy):
        form, material = CLASSES[int(cls)], MATERIAL_GUESS
        name = f"{form}_{material}"
        volume = None
        if len(poly) >= 3:
            mask = np.zeros((h, w), np.uint8)
            cv2.fillPoly(mask, [poly.round().astype(np.int32)], 1)
            raw = raw_volume_ml(depth, mask > 0, plane)
            if raw and raw > 1:
                volume = raw
        if volume is None:
            volume = TYPICAL_VOLUME
        mass = volume * DENSITY.get(name, 1.0)
        x1, y1, x2, y2 = box
        if x2 - x1 < 2 or y2 - y1 < 2:
            continue
        out.append({"box": (x1, y1, x2, y2), "score": conf, "form": form, "material": material,
                    "volume": volume, "mass": mass, "take": int(mass <= HEAVY_G)})
    return out


def to_string(objs):
    """Список объектов -> строка для колонки prediction."""
    if not objs:
        return "none"
    return ";".join(f"{o['box'][0]:.1f} {o['box'][1]:.1f} {o['box'][2]:.1f} {o['box'][3]:.1f} {o['score']:.3f} "
                    f"{o['form']} {o['material']} {o['volume']:.0f} {o['mass']:.0f} {o['take']}" for o in objs)

## 6. Проверка на валидации

Считаем ту же метрику, что на лидерборде, и смотрим на четыре составляющие по отдельности: так видно, какой шаг слабее всего.

In [ ]:
gt, pred = {}, {}
for f in val:
    gt[f["id"]] = [{"box": (a["bbox"][0], a["bbox"][1], a["bbox"][0] + a["bbox"][2], a["bbox"][1] + a["bbox"][3]),
                    "form": a["attributes"]["form"], "material": a["attributes"]["material"],
                    "volume": a["attributes"]["volume_ml"], "take": int(a["attributes"]["mass_g"] <= HEAVY_G)}
                   for a in f["anns"]]
    pred[f["id"]] = predict(f["dir"])
res = metric.components(gt, pred)
print(f"Валидация: Score {res['Score']:.1f} | детекция D {res['D']:.2f}, материал M {res['M']:.2f}, "
      f"объём V {res['V']:.2f}, решение S {res['S']:.2f}")

## 7. Файл с ответами для теста

In [ ]:
test_dirs = sorted(p for p in (DATA / "real" / "test").iterdir() if p.is_dir())
rows = [{"image_id": d.name, "prediction": to_string(predict(d))} for d in test_dirs]
pd.DataFrame(rows).to_csv("submission.csv", index=False)
print(f"Готово: submission.csv, кадров {len(rows)}, с найденными объектами {sum(r['prediction'] != 'none' for r in rows)}")

## Что можно улучшить

- **Детектор.** Больше эпох, модель крупнее (`yolo11s-seg.pt`), аугментации.
- **Синтетика.** В `dataset/synth/train` лежит 600 размеченных кадров, которые бейзлайн не использует. Только следите, чтобы предметы из вашей валидации не попали в обучение через синтетику.
- **Материал.** Сейчас он не определяется вообще, а это пятая часть балла, и от него зависит масса. Можно учить детектор сразу на пары «форма + материал» или обучить отдельный классификатор на вырезках объектов.
- **Глубина в детекции.** Детектор смотрит только на цвет. Высота над полом могла бы стать четвёртым каналом или отсеивать ложные срабатывания.
- **Объём.** Камера видит предмет только спереди и сверху, а у тонких предметов шум глубины сравним с их толщиной. Сравните «сырой» объём с настоящим на обучающих кадрах: ошибка систематическая или случайная? Можно ли её поправить?
- **Стекло и блестящий металл.** У них в глубине дыры. Что делать, когда глубины нет?
- **Решение «забрать / объехать».** Массу через плотность класса не угадать для полной и пустой бутылки. Можно ли предсказывать решение напрямую?
- **Анализ ошибок.** Выведите кадры валидации, где модель ошиблась сильнее всего. Это пригодится на защите.